# =====================================================
# NOTEBOOK INFORMATION
# =====================================================

# Privacy-Preserving Federated Network Intrusion Detection System

## Notebook 05 - Baseline Model Comparison

### Objectives

- Load the feature-engineered dataset
- Split data into training and testing sets
- Train multiple classification models
- Compare model performance
- Select the best model for Federated Learning

In [1]:
# =====================================================
# IMPORT LIBRARIES
# =====================================================

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# =====================================================
# PROJECT CONFIGURATION
# =====================================================

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "CICIDS2017_feature_engineered.csv"
)

RESULTS_PATH = PROJECT_ROOT / "results"
MODELS_PATH = RESULTS_PATH / "models"
METRICS_PATH = RESULTS_PATH / "metrics"
FIGURES_PATH = RESULTS_PATH / "figures"

for path in [
    MODELS_PATH,
    METRICS_PATH,
    FIGURES_PATH
]:
    path.mkdir(parents=True, exist_ok=True)

print(DATA_PATH)

c:\MyFiles\Project\Privacy-Preserving-Federated-NIDS\data\processed\CICIDS2017_feature_engineered.csv


In [3]:
# =====================================================
# LOAD FEATURE-ENGINEERED DATASET
# =====================================================

df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")

df.head()

Dataset Shape: (2520798, 44)


,Destination_Port,Init_Win_bytes_backward,Max_Packet_Length,Bwd_Header_Length,Fwd_Packet_Length_Max,Bwd_Packet_Length_Max,Packet_Length_Mean,Flow_IAT_Mean,Fwd_IAT_Std,min_seg_size_forward,...,Active_Mean,Fwd_Packet_Length_Min,ACK_Flag_Count,Min_Packet_Length,Down_Up_Ratio,URG_Flag_Count,Active_Max,Fwd_PSH_Flags,Idle_Std,Label
0,2.428596,-0.249846,-0.498541,0.001660,-0.297774,-0.478315,-0.580026,-0.308793,-0.361725,0.002698,...,-0.13337,-0.217169,1.485409,-0.423387,-1.006882,-0.335907,-0.158458,-0.226182,-0.11608,0
1,2.438537,-0.221049,-0.498541,0.001673,-0.297774,-0.475372,-0.580026,-0.308771,-0.361725,0.002698,...,-0.13337,-0.217169,1.485409,-0.423387,0.430602,2.977016,-0.158458,-0.226182,-0.11608,0
2,2.438590,-0.221049,-0.498541,0.001673,-0.297774,-0.475372,-0.580026,-0.308783,-0.361725,0.002698,...,-0.13337,-0.217169,1.485409,-0.423387,0.430602,2.977016,-0.158458,-0.226182,-0.11608,0
3,1.974744,-0.212869,-0.498541,0.001673,-0.297774,-0.475372,-0.580026,-0.308787,-0.361725,0.002698,...,-0.13337,-0.217169,1.485409,-0.423387,0.430602,2.977016,-0.158458,-0.226182,-0.11608,0
4,2.428491,-0.249846,-0.498541,0.001660,-0.297774,-0.478315,-0.580026,-0.308793,-0.361725,0.002698,...,-0.13337,-0.217169,1.485409,-0.423387,-1.006882,-0.335907,-0.158458,-0.226182,-0.11608,0


In [4]:
# =====================================================
# SEPARATE FEATURES AND TARGET
# =====================================================

X = df.drop(columns=["Label"])
y = df["Label"]

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")
print(f"Classes: {y.nunique()}")

Features: (2520798, 43)
Target: (2520798,)
Classes: 15


In [5]:
# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 2016638
Testing samples : 504160


In [6]:
# =====================================================
# DEFINE BASELINE MODELS
# =====================================================

models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                n_jobs=-1
            )
        )
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    )
}

print("Models defined:")
for name in models:
    print("-", name)

Models defined:
- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost


In [7]:
# =====================================================
# TRAIN BASELINE MODELS
# =====================================================

results = []
trained_models = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    trained_models[name] = model

    print(f"Accuracy : {accuracy:.4f}")
    print(f"F1 Score : {f1:.4f}")


Training Logistic Regression...
Accuracy : 0.8300
F1 Score : 0.8838

Training Decision Tree...
Accuracy : 0.9983
F1 Score : 0.9983

Training Random Forest...
Accuracy : 0.9984
F1 Score : 0.9984

Training XGBoost...
Accuracy : 0.9988
F1 Score : 0.9988


In [8]:
# =====================================================
# MODEL COMPARISON
# =====================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
3,XGBoost,0.998838,0.998799,0.998838,0.998792
2,Random Forest,0.998358,0.998558,0.998358,0.998421
1,Decision Tree,0.998266,0.998269,0.998266,0.998267
0,Logistic Regression,0.829969,0.964235,0.829969,0.883810
